In [3]:
# %% [markdown]
# # Batch Normalization 实验 - VGG-A
# 作业完整版本，可直接运行

# %%
import matplotlib.pyplot as plt
import numpy as np
import torch
import os
import random
from torch import nn
from tqdm.notebook import tqdm
from IPython import display

# 关键修复：Jupyter 必须用这个路径
module_path = os.getcwd()  # 👈 修复这里
home_path = module_path
figures_path = os.path.join(home_path, 'reports', 'figures')
models_path = os.path.join(home_path, 'reports', 'models')

os.makedirs(figures_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)

# 使用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用设备:", device)

# 加载模型 & 数据
from models.vgg import VGG_A, VGG_A_BatchNorm
from data.loaders import get_cifar_loader

train_loader = get_cifar_loader(train=True, num_workers=0)
val_loader = get_cifar_loader(train=False, num_workers=0)

# 准确率
def get_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, pred = torch.max(outputs.data, 1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

# 随机种子
def set_random_seeds(seed_value=2020):
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    random.seed(seed_value)

# 训练函数
def train(model, optimizer, criterion, epochs_n=20):
    model.to(device)
    losses_list = []
    val_acc_curve = []

    for epoch in range(epochs_n):
        model.train()
        total_loss = 0
        loss_list_epoch = []

        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs_n}"):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            loss_list_epoch.append(loss.item())

        avg_loss = total_loss / len(train_loader)
        val_acc = get_accuracy(model, val_loader)
        losses_list.append(loss_list_epoch)
        val_acc_curve.append(val_acc)

        print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

    return losses_list, val_acc_curve

# %%
# 开始训练
set_random_seeds(2020)
epo = 20
lr = 0.001

print("\n===== 训练 VGG_A without BN =====")
model_no_bn = VGG_A()
opt_no_bn = torch.optim.Adam(model_no_bn.parameters(), lr=lr)
loss_no_bn, val_no_bn = train(model_no_bn, opt_no_bn, nn.CrossEntropyLoss(), epo)

print("\n===== 训练 VGG_A with BN =====")
model_bn = VGG_A_BatchNorm()
opt_bn = torch.optim.Adam(model_bn.parameters(), lr=lr)
loss_bn, val_bn = train(model_bn, opt_bn, nn.CrossEntropyLoss(), epo)

# %%
# 画 Loss Landscape
def get_min_max(losses):
    min_curve = []
    max_curve = []
    for i in range(epo):
        step = [ls[i] for ls in losses if i < len(ls)]
        if step:
            min_curve.append(min(step))
            max_curve.append(max(step))
    return min_curve, max_curve

min_no, max_no = get_min_max(loss_no_bn)
min_bn, max_bn = get_min_max(loss_bn)

def plot_landscape(min_curve, max_curve, title, save_path):
    plt.figure(figsize=(10, 5))
    plt.fill_between(range(len(min_curve)), min_curve, max_curve, alpha=0.5)
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss Range")
    plt.savefig(save_path)
    plt.show()

plot_landscape(min_no, max_no, "VGG-A without BN", os.path.join(figures_path, "loss_no_bn.png"))
plot_landscape(min_bn, max_bn, "VGG-A with BN", os.path.join(figures_path, "loss_bn.png"))

# %%
# 画对比图
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot([np.mean(ls) for ls in loss_no_bn], label="No BN")
plt.plot([np.mean(ls) for ls in loss_bn], label="BN")
plt.title("Loss Curve")
plt.legend()

plt.subplot(1,2,2)
plt.plot(val_no_bn, label="No BN")
plt.plot(val_bn, label="BN")
plt.title("Accuracy Curve")
plt.legend()
plt.show()

print("\n✅ 全部完成！图片保存在：", figures_path)

使用设备: cpu

===== 训练 VGG_A without BN =====


Epoch 1/20:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch  1 | Loss: 2.0641 | Val Acc: 0.2735


Epoch 2/20:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch  2 | Loss: 1.6940 | Val Acc: 0.4244


...

Epoch 19/20:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch 19 | Loss: 0.0552 | Val Acc: 0.8147


Epoch 20/20:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch 20 | Loss: 0.0522 | Val Acc: 0.8217

✅ 全部完成！图片保存在： C:\Users\金力铖\Desktop\IAF\2026\深度学习\project_2\PJ2_2026\PJ2_2026\codes\VGG_BatchNorm\reports\figures


C:\Windows\Temp\ipykernel_24964\3734321051.py:123: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Windows\Temp\ipykernel_24964\3734321051.py:142: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
import matplotlib.pyplot as plt
import os
import numpy as np

# 你的保存路径
figures_path = r"C:\Users\金力铖\Desktop\IAF\2026\深度学习\project_2\PJ2_2026\PJ2_2026\codes\VGG_BatchNorm\reports\figures"
os.makedirs(figures_path, exist_ok=True)

# ======================== 1. 无 BN Loss 曲线 ========================
avg_loss_no_bn = [np.mean(epoch_losses) for epoch_losses in loss_no_bn]

plt.figure(figsize=(8,4))
plt.plot(avg_loss_no_bn, linewidth=2, label='No BN Train Loss')
plt.title('VGG-A without BN - Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(os.path.join(figures_path, 'loss_curve_no_bn.png'), dpi=300, bbox_inches='tight')
plt.close()

# ======================== 2. 无 BN Accuracy 曲线 ========================
plt.figure(figsize=(8,4))
plt.plot(val_no_bn, linewidth=2, label='No BN Val Acc', color='orange')
plt.title('VGG-A without BN - Accuracy Curve')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(os.path.join(figures_path, 'acc_curve_no_bn.png'), dpi=300, bbox_inches='tight')
plt.close()

# ======================== 3. 有 BN Loss 曲线 ========================
avg_loss_bn = [np.mean(epoch_losses) for epoch_losses in loss_bn]

plt.figure(figsize=(8,4))
plt.plot(avg_loss_bn, linewidth=2, label='With BN Train Loss')
plt.title('VGG-A with BN - Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(os.path.join(figures_path, 'loss_curve_bn.png'), dpi=300, bbox_inches='tight')
plt.close()

# ======================== 4. 有 BN Accuracy 曲线 ========================
plt.figure(figsize=(8,4))
plt.plot(val_bn, linewidth=2, label='With BN Val Acc', color='green')
plt.title('VGG-A with BN - Accuracy Curve')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(os.path.join(figures_path, 'acc_curve_bn.png'), dpi=300, bbox_inches='tight')
plt.close()

# ======================== 5. 终极对比图（报告最加分） ========================
plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
plt.plot(avg_loss_no_bn, label='No BN', linewidth=2)
plt.plot(avg_loss_bn, label='With BN', linewidth=2)
plt.title('Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1,2,2)
plt.plot(val_no_bn, label='No BN', linewidth=2)
plt.plot(val_bn, label='With BN', linewidth=2)
plt.title('Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(figures_path, 'final_comparison.png'), dpi=300, bbox_inches='tight')
plt.close()

print("✅ 所有图片已生成！")
print("📍 保存位置：", figures_path)

✅ 所有图片已生成！
📍 保存位置： C:\Users\金力铖\Desktop\IAF\2026\深度学习\project_2\PJ2_2026\PJ2_2026\codes\VGG_BatchNorm\reports\figures
